# PHASE 4: Modeling (Regression & Clustering)

Notebook dùng cùng script R để không lệch bộ đặc trưng hoặc kết quả so với chạy Rscript.
Không giữ các kết luận MAE, số cụm hay cảm xúc viết sẵn từ phiên bản cũ.


In [1]:
# Tìm gốc dự án từ thư mục notebook hoặc thư mục mở Jupyter.
current <- normalizePath(getwd(), winslash = "/")
repeat {
  candidates <- c(current, file.path(current, "DTA301-NYC-Airbnb-Analysis"))
  matched <- candidates[file.exists(file.path(candidates, "code", "analysis_helpers.R"))]
  if (length(matched)) { root <- matched[1]; break }
  if (dirname(current) == current) stop("Hãy mở notebook từ bên trong dự án.")
  current <- dirname(current)
}
source(file.path(root, "code", "analysis_helpers.R"), encoding = "UTF-8")


## 1. Phân cụm (K-Means)

Mã hóa loại phòng, chuẩn hóa biến số; thử K=2..8 và chọn theo Silhouette trên mẫu cố định tối đa 2.000 listing.
Mô hình cuối được khớp trên toàn bộ listings. Xem `kmeans_k_selection.csv`, `cluster_profiles.csv` để diễn giải cụm theo kết quả thật.

## 2. Dự đoán giá (Regression - Random Forest)

Dùng vị trí (borough và neighbourhood), loại phòng, sức chứa, stay_type, số lượng/tần suất reviews và availability.
Chia 80/20 trước khi mã hóa/điền thiếu. So sánh MAE, RMSE và R² với baseline trung vị train.
Giá dự đoán là giá niêm yết tham khảo; chưa chứng minh mức giá tối ưu doanh thu.


In [2]:
source(file.path(root, "code", "04_Modeling", "04_Modeling.R"), encoding = "UTF-8")



Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union




K được chọn: 6 
# A tibble: 6 × 6
  cluster     n median_price median_reviews mean_latitude mean_longitude
  <fct>   <int>        <dbl>          <dbl>         <dbl>          <dbl>
1 1        1186        865.               2          40.7          -74.0
2 2        3893        170.              19          40.7          -73.9
3 3        9061         99.8              7          40.7          -73.9
4 4        6562        230.               3          40.8          -74.0
5 5         457        432                0          40.7          -74.0
6 6         191         54.0              1          40.7          -73.9
KẾT QUẢ DỰ ĐOÁN GIÁ:
                  model n_train n_test       MAE     RMSE          R2 seed
1         Random Forest   17080   4270  99.11097 177.3409  0.47282609   42
2 Train median baseline   17080   4270 147.61092 255.5905 -0.09502939   42
Kết quả holdout mô tả khả năng dự đoán giá niêm yết; chưa chứng minh giá tối ưu hay doanh thu.


## 3. Đánh giá Thái độ khách hàng (Sentiment Analysis)

Chạy `01b_clean_reviews.R` trước để bỏ HTML và chỉ giữ reviews tiếng Anh bằng cld2.
TF-IDF dùng tài liệu theo listing, so sánh rating=5,0 và rating<4,5; kích thước nhóm thực tế được xuất riêng.
Cảm xúc syuzhet chỉ tính trên mẫu tối đa 5.000 reviews, là phân tích bổ sung có giới hạn từ điển và lấy mẫu.
Nếu dùng nguồn reviews đã lọc cũ, các bình luận dưới bốn từ đã mất không thể khôi phục; xem `reviews_preparation.csv`.


In [3]:
source(file.path(root, "code", "04_Modeling", "04_Sentiment_Analysis.R"), encoding = "UTF-8")


Hoàn thành! Thống kê kết quả:
# A tibble: 3 × 3
  sentiment_label     n proportion
  <chr>           <int>      <dbl>
1 Negative           80     0.016 
2 Neutral            47     0.0094
3 Positive         4873     0.975 
Đã xuất TF-IDF, đối chiếu hai nhóm rating và kết quả cảm xúc bổ sung.
